# R3 - Grand-average ERP across subjects

Select the repository `.venv` kernel and Run All. This notebook reads the
released v2 epochs, without repeating preprocessing or changing the data.
It averages trials within each subject first, then averages equally across
subjects. Shading is mean +/- one SEM **across subjects**, not across trials.

Outputs: ERP at Pz/CPz/Cz, target-minus-standard difference waves, group
summary CSV and per-subject epoch counts. Figures are a draft ERP part of F2;
topomaps and P300 peak measurements remain separate next steps.
Visual QC approval from R2 remains pending in the current release log.


In [ ]:
import sys

import matplotlib.pyplot as plt
from IPython.display import display

from eegproj.config import find_root, load_config
from eegproj.evaluation.erp import collect_subject_erps, summarise_erps

ROOT = find_root()
CONFIG = load_config(ROOT)
CHANNELS = ("Pz", "CPz", "Cz")

## 1. Average trials separately within each subject

Only subjects with pipeline_version=2 and excluded_subject=False in the
release log are loaded. Sampling rate, epoch interval, baseline, channels,
metadata and per-condition counts are checked. Each subject is read in turn
to limit memory use. This is a structure check, not a visual QC review.


In [ ]:
subjects, times, subject_erps, epoch_counts = collect_subject_erps(ROOT, CONFIG, CHANNELS)
print("Number of subjects:", len(subjects))
display(epoch_counts)
display(epoch_counts["visual_review"].value_counts().rename("n_subjects").to_frame())

## 2. Group mean and SEM

SEM = sample SD across subject ERPs / sqrt(number of subjects).
For the difference wave, subtract standard from target **within each
subject first**, then calculate the group mean and SEM of these paired
differences. Do not subtract the SEMs of two conditions.


In [ ]:
summary = summarise_erps(subject_erps, times, CHANNELS)
output_dir = ROOT / "results" / "erp"
output_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_dir / "grand_average_summary.csv", index=False)
epoch_counts.to_csv(output_dir / "subject_epoch_counts.csv", index=False)
display(summary.head())
print("Saved tables to:", output_dir)

## 3. ERP at Pz, CPz and Cz

Curves show the average across subjects; shaded areas show +/- one SEM.
EEG amplitude is converted from V to microvolts. The two figure formats
(PNG at 300 dpi and PDF) are saved locally under figures/ and are gitignored.


In [ ]:
figure_dir = ROOT / CONFIG["paths"]["figures"]
figure_dir.mkdir(parents=True, exist_ok=True)
conditions = ("standard", "target", "distractor")
colors = {"standard": "tab:blue", "target": "tab:orange", "distractor": "tab:green"}
plt.rcParams.update({"font.size": 11})
fig, axes = plt.subplots(1, len(CHANNELS), figsize=(15, 4.5), sharex=True, sharey=True)
for ax, channel in zip(axes, CHANNELS):
    for condition in conditions:
        wave = summary.loc[summary["channel"].eq(channel) & summary["condition"].eq(condition)]
        time_ms = wave["time_s"].to_numpy() * 1000
        mean = wave["mean_uv"].to_numpy()
        sem = wave["sem_uv"].to_numpy()
        ax.plot(time_ms, mean, color=colors[condition], label=condition)
        ax.fill_between(time_ms, mean - sem, mean + sem, color=colors[condition], alpha=0.18)
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.set(title=channel, xlabel="Time from stimulus onset (ms)")
    ax.set_xlim(times[0] * 1000, times[-1] * 1000)
axes[0].set_ylabel("Amplitude (microvolts)")
axes[-1].legend(fontsize=10)
fig.suptitle(f"Grand-average ERP: {len(subjects)} subjects, mean +/- SEM (visual QC pending)")
fig.tight_layout()
for suffix in ("png", "pdf"):
    path = figure_dir / f"F2_erp_grand_average.{suffix}"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
plt.show()

## 4. Paired target-minus-standard difference waves

These curves provide an ERP reference for R1's model interpretation.
Use the complete group ERP and the forthcoming topomaps before choosing
a P300 measurement interval. Do not treat one positive peak as evidence
that movement-related activity has been ruled out.


In [ ]:
fig, axes = plt.subplots(1, len(CHANNELS), figsize=(15, 4.5), sharex=True, sharey=True)
for ax, channel in zip(axes, CHANNELS):
    wave = summary.loc[summary["channel"].eq(channel) & summary["condition"].eq("target_minus_standard")]
    time_ms = wave["time_s"].to_numpy() * 1000
    mean = wave["mean_uv"].to_numpy()
    sem = wave["sem_uv"].to_numpy()
    ax.plot(time_ms, mean, color="tab:purple", label="target - standard")
    ax.fill_between(time_ms, mean - sem, mean + sem, color="tab:purple", alpha=0.18)
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.set(title=channel, xlabel="Time from stimulus onset (ms)")
    ax.set_xlim(times[0] * 1000, times[-1] * 1000)
axes[0].set_ylabel("Amplitude difference (microvolts)")
axes[-1].legend(fontsize=10)
fig.suptitle(f"Target - standard: {len(subjects)} paired subjects, mean +/- SEM (visual QC pending)")
fig.tight_layout()
for suffix in ("png", "pdf"):
    path = figure_dir / f"F2_difference_wave.{suffix}"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
plt.show()

## Before sharing results

- Confirm that all expected subjects are represented and no cell failed.
- Coordinate pending visual QC with R2 before finalizing report figures.
- Next: topomaps over 250-600 ms as stated in protocol section 9, then
  subject-level P300 amplitude/latency using an explicitly documented interval.
- Keep generated tables and figures in ignored folders; share them through
  the team's shared folder. Use nbstripout when committing notebook code.
